# 🌀 CycloneX Worker — Google Colab (GPU)

Este notebook roda o **CycloneX como Worker**, conectando ao seu **Master** em casa.

## ✅ Antes de começar:
1. Vá em **Runtime → Change runtime type → GPU** (T4 é grátis)
2. Preencha o `MASTER_URL` e o `TOKEN` na célula de configuração
3. Execute as células em ordem

---

## 🔧 Célula 1 — Verificar GPU disponível

In [ ]:
!nvidia-smi
!nvcc --version

## 📥 Célula 2 — Clonar o repositório CycloneX

In [ ]:
import os

# Clona o repositório
!git clone https://github.com/ggsofthouse/cycloneX.git /content/cycloneX

os.chdir('/content/cycloneX')
!ls -la

## ⚙️ Célula 3 — Compilar CUDACyclone no Linux do Colab

> O `.exe` do Windows não funciona no Linux. Precisamos compilar o `.cu` com `nvcc`.

In [ ]:
import subprocess

os.chdir('/content/cycloneX/CUDACyclone-main')

# Detectar compute capability da GPU do Colab
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
    capture_output=True, text=True
)
cc = result.stdout.strip().replace('.', '') if result.returncode == 0 else '75'
print(f'Compute Capability detectada: {cc}')

# Compilar
print('\n🔨 Compilando CUDACyclone.cu (pode levar 2-5 min)...')
build = subprocess.run(
    ['make', f'GPU_ARCHS={cc}', '-j4'],
    capture_output=True, text=True
)
print(build.stdout[-3000:] if len(build.stdout) > 3000 else build.stdout)
if build.returncode != 0:
    print('STDERR:', build.stderr[-2000:])
    raise RuntimeError('❌ Build falhou!')
else:
    print('\n✅ Build concluído com sucesso!')
    !ls -lh CUDACyclone

## 🌐 Célula 4 — CONFIGURAR: coloque aqui o IP do seu Master

> **⚠️ Preencha `MASTER_URL` e `TOKEN` antes de continuar!**

In [ ]:
# ══════════════════════════════════════════
# 👇 EDITE AQUI:
MASTER_URL = "http://SEU_IP_PUBLICO:8080"  # Ex: http://123.45.67.89:8080 ou URL do ngrok
TOKEN      = "SEU_TOKEN_SECRETO_AQUI"      # Mesmo token do config.yaml do Master
# ══════════════════════════════════════════

# Grid e slices recomendados para T4 do Colab
GRID   = "512,1024"
SLICES = 256

print(f'Master URL: {MASTER_URL}')
print(f'Token: {TOKEN[:6]}...{TOKEN[-4:]} (oculto por segurança)')
print(f'Grid: {GRID} | Slices: {SLICES}')

## 📝 Célula 5 — Gerar config.yaml do Worker

In [ ]:
os.chdir('/content/cycloneX')

config_content = f"""job:
  grid: \"{GRID}\"
  slices: {SLICES}
  gpus: \"0\"
  random: false

gpu:
  max_temp_c: 85

telemetry:
  interval: 1

api:
  port: 8081
  token: \"{TOKEN}\"

database:
  path: \"./cyclone.db\"

pool:
  role: \"worker\"
  master_url: \"{MASTER_URL}\"
  block_size_gkeys: 200
  random_in_block: false
"""

with open('config.yaml', 'w') as f:
    f.write(config_content)

print('✅ config.yaml gerado:')
print(config_content)

## 🔗 Célula 6 — Testar conexão com o Master

In [ ]:
import urllib.request, json

try:
    req = urllib.request.Request(
        f"{MASTER_URL}/api/status",
        headers={"Authorization": f"Bearer {TOKEN}"}
    )
    with urllib.request.urlopen(req, timeout=10) as r:
        data = json.loads(r.read())
    print('✅ Master acessível!')
    print(json.dumps(data, indent=2))
except Exception as e:
    print(f'❌ Não foi possível conectar ao Master: {e}')
    print('Verifique o MASTER_URL e se o Master está rodando com a porta aberta.')

## 🚀 Célula 7 — Iniciar o Worker CycloneX

> Esta célula é bloqueante. O worker vai rodar continuamente pedindo blocos ao Master.
>
> ⏱️ **Colab grátis**: até ~12h de GPU por sessão. Após isso, reinicie e rode novamente.

In [ ]:
import subprocess, sys, os

os.chdir('/content/cycloneX')

# Ajustar o caminho do exe compilado
# O cyclone_agent.py procura CUDACyclone.exe — vamos criar um symlink/wrapper
if not os.path.exists('CUDACyclone.exe'):
    os.symlink('/content/cycloneX/CUDACyclone-main/CUDACyclone', 'CUDACyclone.exe')
    print('Symlink CUDACyclone.exe → CUDACyclone criado')

print('🚀 Iniciando CycloneX Worker...')
print(f'   Master: {MASTER_URL}')
print('   Pressione Stop (■) para encerrar\n')

proc = subprocess.Popen(
    [sys.executable, 'cyclone_agent.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

try:
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print('\n⛔ Worker encerrado.')

---

## 📊 Monitoramento

Acompanhe no **dashboard do seu Master** em `http://localhost:8080` → aba **Pool/Workers**.

O Colab aparecerá como worker com o hostname da instância Google.

---

## ⚠️ Limitações do Colab Gratuito

| Item | Colab Free | Colab Pro |
|------|-----------|----------|
| GPU | Tesla T4 (~8 GKeys/s) | A100 (~50 GKeys/s) |
| Tempo máx. | ~12h/sessão | ~24h/sessão |
| Sessões simultâneas | 1 | Até 3 |
| Custo | Grátis | ~$10/mês |

💡 **Dica:** Você pode abrir múltiplas abas do Colab e rodar vários workers ao mesmo tempo com contas Google diferentes!